In [0]:
# Old approach: hardcoded Kafka connection settings (kept for reference only)
# Superseded by Section 1 below, which loads credentials from the Databricks secret scope.
# - bootstrapservers: Confluent Cloud bootstrap server endpoint
# - apikey / apiSecret: API credentials for SASL/PLAIN authentication
# - topic: Kafka topic holding the credit card transactions
bootstrapservers='pkc-41p56.asia-south1.gcp.confluent.cloud:9092'
# apikey='SDUBJ4CPDGHWZ7KZ'
# apiSecret='cflthQclTL86Phpbp4nurlFJwZdbihOTA7JrYBf9H0t+zxAgq8IJ0irA0/pg2moA'
# topic='credit_card_transactions'

In [0]:
# Section 1: Load Kafka connection settings from the Databricks secret scope
# Reads the 'kafka_connection_details' JSON secret from scope 'finguard-scope'
# and unpacks bootstrap servers, API key/secret and topic name.
import json
kafka_connection_json = dbutils.secrets.get(scope = "finguard-scope", key = "kafka_connection_details")
kafka_config = json.loads(kafka_connection_json)
topic = kafka_config['topic']
apiSecret = kafka_config['api_secret']
apikey = kafka_config['api_key']
bootstrapservers = kafka_config['bootstrap_servers']

In [0]:
# Section 2: Build the SASL/PLAIN JAAS config string
# The Kafka client uses this to authenticate to Confluent Cloud with the API key/secret.
# It is passed to Spark as the "kafka.sasl.jaas.config" option.
jasas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{apikey}" password="{apiSecret}";'

In [0]:
# Section 3: Batch read from Kafka
# Reads all messages currently in the topic (earliest -> latest offsets)
# as a one-time batch, authenticating with SASL_SSL + PLAIN.
sample_batch = spark.read.format("kafka") \
  .option("kafka.bootstrap.servers", bootstrapservers) \
  .option("subscribe", topic) \
  .option("kafka.security.protocol", "SASL_SSL") \
   .option("kafka.sasl.mechanism", "PLAIN") \
  .option("kafka.sasl.jaas.config", jasas_config) \
  .option("startingOffsets", "earliest") \
  .load()
display(sample_batch)

key,value,topic,partition,offset,timestamp,timestampType
VFhONDQ1ODk2,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE40NDU4OTYiLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDgyOSIsICJjYXJkX251bWJlciI6ICI1NTg0ODI2NjkxMzYyNDgyIiwgIm1lcmNoYW50X2lkIjogIk1FUjAwMDIiLCAibWVyY2hhbnRfbmFtZSI6ICJUYXN0eSBUYWJsZSAyIiwgIm1lcmNoYW50X2NhdGVnb3J5IjogIlJlc3RhdXJhbnQiLCAiYW1vdW50IjogMjAzMzcuNjksICJjdXJyZW5jeSI6ICJJTlIiLCAidHJhbnNhY3Rpb25fdHlwZSI6ICJQVVJDSEFTRSIsICJwYXltZW50X2NoYW5uZWwiOiAiSU5URVJORVRfQkFOS0lORyIsICJkZXZpY2VfaWQiOiAiREVWMDgyOSIsICJjaXR5IjogIkh5ZGVyYWJhZCIsICJjb3VudHJ5IjogIkluZGlhIiwgInRyYW5zYWN0aW9uX3RpbWVzdGFtcCI6ICIyMDI2LTExLTEwVDE5OjQ4OjU5WiIsICJpc19pbnRlcm5hdGlvbmFsIjogZmFsc2UsICJzdGF0dXMiOiAiQVBQUk9WRUQifQ==,credit_card_transactions,2,0,2026-11-10T19:48:59.786Z,0
VFhONjE3OTg0,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE42MTc5ODQiLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDM5MCIsICJjYXJkX251bWJlciI6ICI0NzU0ODIxNDU2NTMyODYzIiwgIm1lcmNoYW50X2lkIjogIk1FUjAxMDciLCAibWVyY2hhbnRfbmFtZSI6ICJUcmF2ZWxFYXNlIDEwNyIsICJtZXJjaGFudF9jYXRlZ29yeSI6ICJUcmF2ZWwiLCAiYW1vdW50IjogNzI0OTIuOTksICJjdXJyZW5jeSI6ICJJTlIiLCAidHJhbnNhY3Rpb25fdHlwZSI6ICJQVVJDSEFTRSIsICJwYXltZW50X2NoYW5uZWwiOiAiUE9TIiwgImRldmljZV9pZCI6ICJERVYwMzkwIiwgImNpdHkiOiAiTHVja25vdyIsICJjb3VudHJ5IjogIkluZGlhIiwgInRyYW5zYWN0aW9uX3RpbWVzdGFtcCI6ICIyMDI2LTExLTEwVDE5OjQ5OjAxWiIsICJpc19pbnRlcm5hdGlvbmFsIjogdHJ1ZSwgInN0YXR1cyI6ICJBUFBST1ZFRCJ9,credit_card_transactions,2,1,2026-11-10T19:49:01.194Z,0
VFhONTQ1MDkw,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE41NDUwOTAiLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDg5MiIsICJjYXJkX251bWJlciI6ICI0NzMzOTMwNDkzNjY1Mzg2IiwgIm1lcmNoYW50X2lkIjogIk1FUjAxMTgiLCAibWVyY2hhbnRfbmFtZSI6ICJFdmVudEh1YiAxMTgiLCAibWVyY2hhbnRfY2F0ZWdvcnkiOiAiRW50ZXJ0YWlubWVudCIsICJhbW91bnQiOiA1MTc3NS4wLCAiY3VycmVuY3kiOiAiSU5SIiwgInRyYW5zYWN0aW9uX3R5cGUiOiAiUFVSQ0hBU0UiLCAicGF5bWVudF9jaGFubmVsIjogIklOVEVSTkVUX0JBTktJTkciLCAiZGV2aWNlX2lkIjogIkRFVjA4OTIiLCAiY2l0eSI6ICJIeWRlcmFiYWQiLCAiY291bnRyeSI6ICJJbmRpYSIsICJ0cmFuc2FjdGlvbl90aW1lc3RhbXAiOiAiMjAyNi0xMS0xMFQxOTo0OTowNFoiLCAiaXNfaW50ZXJuYXRpb25hbCI6IGZhbHNlLCAic3RhdHVzIjogIkFQUFJPVkVEIn0=,credit_card_transactions,2,2,2026-11-10T19:49:04.008Z,0
VFhONjQ4OTY3,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE42NDg5NjciLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDU1MyIsICJjYXJkX251bWJlciI6ICI1OTQwNjI2MDc4NzAxMDQwIiwgIm1lcmNoYW50X2lkIjogIk1FUjAwMDIiLCAibWVyY2hhbnRfbmFtZSI6ICJUYXN0eSBUYWJsZSAyIiwgIm1lcmNoYW50X2NhdGVnb3J5IjogIlJlc3RhdXJhbnQiLCAiYW1vdW50IjogNTEwNjYuNDQsICJjdXJyZW5jeSI6ICJJTlIiLCAidHJhbnNhY3Rpb25fdHlwZSI6ICJPTkxJTkUiLCAicGF5bWVudF9jaGFubmVsIjogIk1PQklMRV9BUFAiLCAiZGV2aWNlX2lkIjogIkRFVjY5NDciLCAiY2l0eSI6ICJIeWRlcmFiYWQiLCAiY291bnRyeSI6ICJJbmRpYSIsICJ0cmFuc2FjdGlvbl90aW1lc3RhbXAiOiAiMjAyNi0xMS0xMFQxOTo0OTowNFoiLCAiaXNfaW50ZXJuYXRpb25hbCI6IGZhbHNlLCAic3RhdHVzIjogIkFQUFJPVkVEIn0=,credit_card_transactions,2,3,2026-11-10T19:49:04.410Z,0
VFhOODIyMjk0,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE44MjIyOTQiLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDQ2MyIsICJjYXJkX251bWJlciI6ICI1NjA1MjMzMDI3NjQyMDg5IiwgIm1lcmNoYW50X2lkIjogIk1FUjAwNDgiLCAibWVyY2hhbnRfbmFtZSI6ICJGcmVzaE1hcnQgNDgiLCAibWVyY2hhbnRfY2F0ZWdvcnkiOiAiR3JvY2VyeSIsICJhbW91bnQiOiA3MTU3OC4zOSwgImN1cnJlbmN5IjogIklOUiIsICJ0cmFuc2FjdGlvbl90eXBlIjogIlBVUkNIQVNFIiwgInBheW1lbnRfY2hhbm5lbCI6ICJQT1MiLCAiZGV2aWNlX2lkIjogIkRFVjEzNDQiLCAiY2l0eSI6ICJKYWlwdXIiLCAiY291bnRyeSI6ICJJbmRpYSIsICJ0cmFuc2FjdGlvbl90aW1lc3RhbXAiOiAiMjAyNi0xMS0xMFQxOTo0OTowNVoiLCAiaXNfaW50ZXJuYXRpb25hbCI6IGZhbHNlLCAic3RhdHVzIjogIkFQUFJPVkVEIn0=,credit_card_transactions,2,4,2026-11-10T19:49:05.817Z,0
VFhOMTUxMDUx,eyJ0cmFuc2FjdGlvbl9pZCI6ICJUWE4xNTEwNTEiLCAiY3VzdG9tZXJfaWQiOiAiQ1VTVDAwMDU1OSIsICJjYXJkX251bWJlciI6ICI1NDYxNzY3OTI5ODg0MjU0IiwgIm1lcmNoYW50X2lkIjogIk1FUjAwMzYiLCAibWVyY2hhbnRfbmFtZSI6ICJDaXR5IFBhbnRyeSAzNiIsICJtZXJjaGFudF9jYXRlZ29yeSI6ICJHcm9jZXJ5IiwgImFtb3VudCI6IDk5NTg4LjUxLCAiY3VycmVuY3kiOiAiSU5SIiwgInRyYW5zYWN0aW9uX3R5cGUiOiAiT05MSU5FIiwgInBheW1lbnRfY2hhbm5lbCI6ICJQT1MiLCAiZGV2aWNlX2lkIjogIkRFVjQ1ODIiLCAiY2l0eSI6ICJKYWlwdXIiLCAiY291bnRyeSI6ICJJbmRpYSIsICJ0cmFuc2FjdGlvbl90aW1lc3RhbXAiOiAiMjAyNi0xMS0xMFQxOTo0ODo1OVoiLCAiaXNfaW50ZXJuYXRpb25h

In [0]:
# Sanity check: how many messages were read from the topic
sample_batch.count()

45

In [0]:
# Section 4: Inspect the raw Kafka messages
# Selects the Kafka metadata columns (key, value, topic, partition, offset,
# timestamp) and casts key/value to readable strings.
from pyspark.sql.functions import from_json, col, unbase64, base64, split
from pyspark.sql.types import StructField, StructType, FloatType, IntegerType, StringType
parsed_batch = sample_batch.select(
  col("key").cast("string"),
  col("value").cast("string"),
  col("topic"),
  col("partition"),
  col("offset"),
  col("timestamp"),
  col("timestampType")
)

In [0]:
# Display the inspected batch messages
display(parsed_batch)

key,value,topic,partition,offset,timestamp,timestampType
TXN445896,"{""transaction_id"": ""TXN445896"", ""customer_id"": ""CUST000829"", ""card_number"": ""5584826691362482"", ""merchant_id"": ""MER0002"", ""merchant_name"": ""Tasty Table 2"", ""merchant_category"": ""Restaurant"", ""amount"": 20337.69, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0829"", ""city"": ""Hyderabad"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:48:59Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,2,0,2026-11-10T19:48:59.786Z,0
TXN617984,"{""transaction_id"": ""TXN617984"", ""customer_id"": ""CUST000390"", ""card_number"": ""4754821456532863"", ""merchant_id"": ""MER0107"", ""merchant_name"": ""TravelEase 107"", ""merchant_category"": ""Travel"", ""amount"": 72492.99, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV0390"", ""city"": ""Lucknow"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:49:01Z"", ""is_international"": true, ""status"": ""APPROVED""}",credit_card_transactions,2,1,2026-11-10T19:49:01.194Z,0
TXN545090,"{""transaction_id"": ""TXN545090"", ""customer_id"": ""CUST000892"", ""card_number"": ""4733930493665386"", ""merchant_id"": ""MER0118"", ""merchant_name"": ""EventHub 118"", ""merchant_category"": ""Entertainment"", ""amount"": 51775.0, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0892"", ""city"": ""Hyderabad"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:49:04Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,2,2,2026-11-10T19:49:04.008Z,0
TXN648967,"{""transaction_id"": ""TXN648967"", ""customer_id"": ""CUST000553"", ""card_number"": ""5940626078701040"", ""merchant_id"": ""MER0002"", ""merchant_name"": ""Tasty Table 2"", ""merchant_category"": ""Restaurant"", ""amount"": 51066.44, ""currency"": ""INR"", ""transaction_type"": ""ONLINE"", ""payment_channel"": ""MOBILE_APP"", ""device_id"": ""DEV6947"", ""city"": ""Hyderabad"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:49:04Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,2,3,2026-11-10T19:49:04.410Z,0
TXN822294,"{""transaction_id"": ""TXN822294"", ""customer_id"": ""CUST000463"", ""card_number"": ""5605233027642089"", ""merchant_id"": ""MER0048"", ""merchant_name"": ""FreshMart 48"", ""merchant_category"": ""Grocery"", ""amount"": 71578.39, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV1344"", ""city"": ""Jaipur"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:49:05Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,2,4,2026-11-10T19:49:05.817Z,0
TXN151051,"{""transaction_id"": ""TXN151051"", ""customer_id"": ""CUST000559"", ""card_number"": ""5461767929884254"", ""merchant_id"": ""MER0036"", ""merchant_name"": ""City Pantry 36"", ""merchant_category"": ""Grocery"", ""amount"": 99588.51, ""currency"": ""INR"", ""transaction_type"": ""ONLINE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV4582"", ""city"": ""Jaipur"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:48:59Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,1,0,2026-11-10T19:48:59.183Z,0
TXN236000,"{""transaction_id"": ""TXN236000"", ""customer_id"": ""CUST000285"", ""card_number"": ""4144885015569904"", ""merchant_id"": ""MER0059"", ""merchant_name"": ""Globe Trek 59"", ""merchant_category"": ""Travel"", ""amount"": 16607.11, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV0285"", ""city"": ""Chennai"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T19:48:59Z

In [0]:
# Section 5: Write the inspected batch to the bronze Delta table
parsed_batch.write.saveAsTable("finguard.bronze.transactions_batch_test")

In [0]:
# Section 6: Streaming read from Kafka
# Same connection settings as the batch read, but as a streaming source
# so new messages are picked up incrementally.
streaming_df= spark.readStream.format("kafka") \
  .option("kafka.bootstrap.servers", bootstrapservers) \
  .option("subscribe", topic) \
  .option("kafka.security.protocol", "SASL_SSL") \
   .option("kafka.sasl.mechanism", "PLAIN") \
  .option("kafka.sasl.jaas.config", jasas_config) \
  .option("startingOffsets", "earliest") \
  .load()

In [0]:
# Section 7: Inspect the streaming messages
# Same column selection/casting as the batch inspection (Section 4),
# applied to the streaming DataFrame.
from pyspark.sql.functions import from_json, col, unbase64, base64, split
parsed_streaming_df = streaming_df.select(
  col("key").cast("string"),
  col("value").cast("string"),
  col("topic"),
  col("partition"),
  col("offset"),
  col("timestamp"),
  col("timestampType")
)

In [0]:
# Section 8: Write the stream to the bronze Delta table
# - append mode, Delta format
# - checkpointLocation tracks processed offsets (exactly-once, resumable)
# - availableNow trigger: processes all pending messages once, then stops
# Target table: finguard.bronze.transactions_streaming_test
streaming_query = parsed_streaming_df.writeStream.format("delta")\
.outputMode("append")\
.option("checkpointLocation", "/Volumes/finguard/source/transactions/checkpoint/")\
.trigger(availableNow=True)\
.toTable("finguard.bronze.transactions_streaming_test")\

print("quaery started with query ID : ", streaming_query.id)

quaery started with query ID :  166f8c58-d395-4758-9f64-960dab3ea25f


In [0]:
%sql
-- Section 9: Inspect the streaming bronze table
select * from finguard.bronze.transactions_streaming_test

key,value,topic,partition,offset,timestamp,timestampType
TXN943764,"{""transaction_id"": ""TXN943764"", ""customer_id"": ""CUST000559"", ""card_number"": ""5461767929884254"", ""merchant_id"": ""MER0036"", ""merchant_name"": ""City Pantry 36"", ""merchant_category"": ""Grocery"", ""amount"": 4233.3, ""currency"": ""INR"", ""transaction_type"": ""ONLINE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV4582"", ""city"": ""Jaipur"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:27Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,4,9,2026-11-10T20:58:27.468Z,0
TXN109684,"{""transaction_id"": ""TXN109684"", ""customer_id"": ""CUST000666"", ""card_number"": ""4500507026086988"", ""merchant_id"": ""MER0158"", ""merchant_name"": ""Tasty Table 158"", ""merchant_category"": ""Restaurant"", ""amount"": 55468.37, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""MOBILE_APP"", ""device_id"": ""DEV0666"", ""city"": ""Ahmedabad"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:27Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,4,10,2026-11-10T20:58:27.873Z,0
TXN648255,"{""transaction_id"": ""TXN648255"", ""customer_id"": ""CUST000128"", ""card_number"": ""4646471543799377"", ""merchant_id"": ""MER0146"", ""merchant_name"": ""Tasty Table 146"", ""merchant_category"": ""Restaurant"", ""amount"": 42290.8, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0128"", ""city"": ""Bengaluru"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:28Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,4,11,2026-11-10T20:58:28.887Z,0
TXN934499,"{""transaction_id"": ""TXN934499"", ""customer_id"": ""CUST000547"", ""card_number"": ""4614878245569640"", ""merchant_id"": ""MER0095"", ""merchant_name"": ""TripVista 95"", ""merchant_category"": ""Travel"", ""amount"": 51341.49, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""POS"", ""device_id"": ""DEV0547"", ""city"": ""Jaipur"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:29Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,4,12,2026-11-10T20:58:29.892Z,0
TXN905280,"{""transaction_id"": ""TXN905280"", ""customer_id"": ""CUST000480"", ""card_number"": ""4650167318826176"", ""merchant_id"": ""MER0098"", ""merchant_name"": ""Spice Avenue 98"", ""merchant_category"": ""Restaurant"", ""amount"": 39617.05, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0480"", ""city"": ""Lucknow"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:31Z"", ""is_international"": true, ""status"": ""APPROVED""}",credit_card_transactions,4,13,2026-11-10T20:58:31.502Z,0
TXN230491,"{""transaction_id"": ""TXN230491"", ""customer_id"": ""CUST000699"", ""card_number"": ""5352635602293949"", ""merchant_id"": ""MER0099"", ""merchant_name"": ""Skyline Hotel 99"", ""merchant_category"": ""Hotel"", ""amount"": 67761.33, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0699"", ""city"": ""Hyderabad"", ""country"": ""India"", ""transaction_timestamp"": ""2026-11-10T20:58:31Z"", ""is_international"": false, ""status"": ""APPROVED""}",credit_card_transactions,4,14,2026-11-10T20:58:31.703Z,0
TXN202242,"{""transaction_id"": ""TXN202242"", ""customer_id"": ""CUST000892"", ""card_number"": ""4733930493665386"", ""merchant_id"": ""MER0118"", ""merchant_name"": ""EventHub 118"", ""merchant_category"": ""Entertainment"", ""amount"": 62287.84, ""currency"": ""INR"", ""transaction_type"": ""PURCHASE"", ""payment_channel"": ""INTERNET_BANKING"", ""device_id"": ""DEV0892"", ""city"": ""Hyderabad"", ""country"": ""India"", ""tran